## Sel 1: Import Pustaka

In [1]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb

from sklearn.multioutput import MultiOutputRegressor

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [2]:
# 1. BACA DATASET (Ganti dengan nama file aslimu)
df = pd.read_csv('dataset_polutan_jatim60.csv')

# --- BAGIAN INI HARUS DISESUAIKAN DENGAN NAMA KOLOM ASLI DI EXCEL-MU ---
kolom_waktu = 'Waktu'       # Kolom yang berisi tanggal dan jam
kolom_kota = 'Kota'         # Kolom nama kota/kabupaten
polutan = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)','SO2 (µg/m³)', 'CO (µg/m³)', 'NO2 (µg/m³)', 'Ozon (µg/m³)'] # Pastikan namanya persis sama
# -----------------------------------------------------------------------

# 2. Ubah format teks menjadi tipe Datetime yang dikenali Python
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])

# 3. Jadikan Waktu dan Kota sebagai Index agar mudah di-resample
df.set_index(kolom_waktu, inplace=True)

# 4. Resample: Hitung rata-rata polutan per HARI (huruf 'D' = Daily) untuk setiap Kota
# Cara ini akan memadatkan 24 baris (per jam) menjadi 1 baris per hari
df_daily = df.groupby(kolom_kota)[polutan].resample('D').mean().reset_index()

# Hapus baris yang kosong (misal ada hari dimana sensor mati total)
df_daily = df_daily.dropna()

print(f"Bentuk data setelah dijadikan rata-rata harian: {df_daily.shape}")
df_daily.head()

Bentuk data setelah dijadikan rata-rata harian: (2318, 8)


,Kota,Waktu,PM2.5 (µg/m³),PM10 (µg/m³),SO2 (µg/m³),CO (µg/m³),NO2 (µg/m³),Ozon (µg/m³)
0,Kabupaten Bangkalan,2026-02-22,17.432143,20.432143,1.152143,548.186429,5.871429,40.233571
1,Kabupaten Bangkalan,2026-02-23,13.778750,16.843750,0.712500,427.443750,4.183750,20.073750
2,Kabupaten Bangkalan,2026-02-24,7.332500,9.698125,1.066250,337.213125,3.926875,34.053125
3,Kabupaten Bangkalan,2026-02-25,7.234167,10.883333,1.179167,354.936250,4.758333,21.076667
4,Kabupaten Bangkalan,2026-02-26,2.575417,4.392500,0.875000,180.410833,2.883333,20.656250


## Sel 3: Rekayasa Fitur (Sliding Window 3 Hari & Target Besok)

In [3]:
# Fungsi untuk membuat Sliding Window, Fitur Temporal, dan Rolling Stats
def create_advanced_features(df_kota, n_lags=3):
    df_temp = df_kota.copy()
    
    # 🌟 FITUR BARU 1: KONTEKS WAKTU (TEMPORAL)
    # Mengajarkan AI tentang konsep musim dan hari libur
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in polutan:
        # --- FITUR LAMA: HISTORY (t-1, t-2, t-3) ---
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
        # 🌟 FITUR BARU 2: TREN JANGKA MENENGAH (ROLLING STATS)
        # Melihat rata-rata dan puncak polusi dalam 3 hari ke belakang.
        # Wajib di-shift(1) agar tidak tidak curang melihat data hari ini.
        df_temp[f'{p}_RollMean_3'] = df_temp[p].shift(1).rolling(window=3).mean()
        df_temp[f'{p}_RollMax_3'] = df_temp[p].shift(1).rolling(window=3).max()
        
        # --- TARGET BESOK (t+1) ---
        df_temp[f'TARGET_{p}_Besok'] = df_temp[p].shift(-1)
        
    return df_temp

print("Meracik fitur tingkat lanjut...")

# Aplikasikan fungsi di atas untuk masing-masing kota secara terpisah
df_model = df_daily.groupby(kolom_kota, group_keys=False).apply(create_advanced_features)

# Hapus baris yang memiliki NaN akibat pergeseran history dan rolling
df_model = df_model.dropna().reset_index(drop=True)

# Lakukan One-Hot Encoding untuk kolom Kota
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

# 1. Kumpulkan semua nama kolom target
kolom_target = [col for col in df_model.columns if 'TARGET_' in col]

# 2. Pisahkan tabel jawaban (y)
y = df_model[kolom_target]

# 3. Pisahkan tabel soal (X) 
X = df_model.drop(columns=kolom_target + [kolom_waktu])

print(f"Bentuk tabel awal df_model : {df_model.shape}")
print(f"Dimensi X (Fitur AI BARU!) : {X.shape}")
print(f"Dimensi y (Target AI)      : {y.shape}")

Meracik fitur tingkat lanjut...
Bentuk tabel awal df_model : (2166, 83)
Dimensi X (Fitur AI BARU!) : (2166, 76)
Dimensi y (Target AI)      : (2166, 6)


## Sel 4: Splitting & Training Baseline Model MLFLOW

In [4]:
# 1. Splitting Data (80% Training, 20% Testing) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Set nama eksperimen utama di MLflow
mlflow.set_experiment("Proyek_ISPU_Jatim")

# 2. Persiapan 3 Baseline Model
baselines = {
    "Baseline_LinearRegression": MultiOutputRegressor(LinearRegression()),
    "Baseline_RandomForest": MultiOutputRegressor(RandomForestRegressor(n_estimators=50, random_state=42)),
    "Baseline_KNeighbors": MultiOutputRegressor(KNeighborsRegressor())
}

print("--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---")
for nama_model, model in baselines.items():
    # Membuka sesi pencatatan MLflow untuk setiap model
    with mlflow.start_run(run_name=nama_model):
        
        # Training & Prediksi
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Kalkulasi Metrik Keseluruhan
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Mencatat parameter, metrik, dan artifact model ke MLflow
        mlflow.log_param("algoritma", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        mlflow.sklearn.log_model(model, artifact_path=nama_model)
        
        print(f"✅ {nama_model} berhasil dicatat (RMSE: {rmse:.2f})")

--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---


2026/05/07 08:17:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:17:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_LinearRegression berhasil dicatat (RMSE: 39.08)


2026/05/07 08:19:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:19:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_RandomForest berhasil dicatat (RMSE: 28.32)


2026/05/07 08:19:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:19:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_KNeighbors berhasil dicatat (RMSE: 38.68)


## Sel 5 : Hyperparameter Tuning XGBoost MLFlow

In [5]:
print("Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...")

# 1. Siapkan 'brankas' untuk menyimpan 6 otak AI yang berbeda
model_terbaik_per_polutan = {}

# 2. Ruang Pencarian Grid
param_grid_murni = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# --- JEMBATAN TQDM & JOBLIB ---
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
# ------------------------------------------

total_mae, total_rmse = 0, 0
daftar_kolom_target = y.columns 

# 3. Lakukan Perulangan: Latih 1 AI untuk tiap 1 Polutan
for nama_kolom in daftar_kolom_target:
    
    # Merapikan nama untuk keperluan print
    nama_polutan_mentah = nama_kolom.replace('TARGET_', '').replace('_Besok', '')
    
    # MEMBERSIHKAN NAMA UNTUK MLFLOW
    nama_polutan_bersih = nama_polutan_mentah.split(' (')[0].replace('.', '_')
    
    print(f"\n======================================================")
    print(f"🚀 Mendidik AI Spesialis Khusus: {nama_polutan_mentah}")
    print(f"======================================================")
    
    # Ambil kolom jawaban (y) HANYA untuk gas ini
    y_train_tunggal = y_train[nama_kolom]
    y_test_tunggal = y_test[nama_kolom]
    
    # Inisiasi XGBoost Murni
    model_xgb_dasar = xgb.XGBRegressor(random_state=42)
    
    # Siapkan Mesin Pencari
    grid_search = RandomizedSearchCV(
        estimator=model_xgb_dasar, 
        param_distributions=param_grid_murni, 
        n_iter=15, 
        cv=4,      
        scoring='neg_mean_absolute_error', 
        random_state=42,
        n_jobs=-1, 
        verbose=0  
    )
    
    # [MLFLOW] Membuka Sesi Pencatatan untuk Polutan Spesifik
    with mlflow.start_run(run_name=f"Spesialis_XGBoost_{nama_polutan_bersih}"):
        
        total_fits = grid_search.n_iter * grid_search.cv 
        with tqdm_joblib(tqdm(desc=f"Tuning {nama_polutan_bersih}", total=total_fits, unit="fit")):
            grid_search.fit(X_train, y_train_tunggal)
            
        # Ambil otak terbaiknya
        otak_spesialis = grid_search.best_estimator_
        
        # Simpan ke dalam brankas dictionary
        model_terbaik_per_polutan[nama_kolom] = otak_spesialis
        
        print(f"Kombinasi Terbaik:\n{grid_search.best_params_}")
        
        # Lakukan ujian evaluasi langsung
        y_pred_tunggal = otak_spesialis.predict(X_test)
        
        mae = mean_absolute_error(y_test_tunggal, y_pred_tunggal)
        rmse = np.sqrt(mean_squared_error(y_test_tunggal, y_pred_tunggal))
        rata_rata_asli = y_test_tunggal.mean()
        persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli > 0 else 0
        
        total_mae += mae
        total_rmse += rmse
        
        print(f"  - Rata-rata asli : {rata_rata_asli:.2f}")
        print(f"  - MAE (Meleset)  : {mae:.2f}")
        print(f"  - RMSE           : {rmse:.2f}")
        print(f"  - Error (%)      : {persentase_error:.2f}%")
        
        # [MLFLOW] Catat skor evaluasi ke dashboard
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        
        # [MLFLOW] Catat resep parameter yang menang ke dashboard
        for param_name, param_val in grid_search.best_params_.items():
            mlflow.log_param(param_name, param_val)
            
        # [MLFLOW] Simpan model spesialis ini ke brankas MLflow
        mlflow.sklearn.log_model(otak_spesialis, artifact_path=f"Model_{nama_polutan_bersih}")

print("\n--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---")
print(f"Rata-rata MAE Keseluruhan : {total_mae/6:.2f}")

Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...

🚀 Mendidik AI Spesialis Khusus: PM2.5 (µg/m³)


Tuning PM2_5:   0%|          | 0/60 [00:00<?, ?fit/s]

2026/05/07 08:20:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:20:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 7.99
  - MAE (Meleset)  : 1.80
  - RMSE           : 3.60
  - Error (%)      : 22.55%

🚀 Mendidik AI Spesialis Khusus: PM10 (µg/m³)


Tuning PM10:   0%|          | 0/60 [00:00<?, ?fit/s]

...

2026/05/07 08:22:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:22:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 1.83
  - MAE (Meleset)  : 0.49
  - RMSE           : 1.18
  - Error (%)      : 26.77%

🚀 Mendidik AI Spesialis Khusus: Ozon (µg/m³)


Tuning Ozon:   0%|          | 0/60 [00:00<?, ?fit/s]

2026/05/07 08:23:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:23:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 33.99
  - MAE (Meleset)  : 2.84
  - RMSE           : 4.72
  - Error (%)      : 8.35%

--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---
Rata-rata MAE Keseluruhan : 6.56


## Sel 6: Ekspor Model (Menyimpan Hasil)

In [6]:
# Membungkus KE-ENAM model spesialis dan nama kolom
paket_model = {
    'dict_model_spesialis': model_terbaik_per_polutan, # Berisi 6 model XGBoost
    'fitur': list(X_train.columns)
}

# Menyimpan ke dalam file Pickle
nama_file = 'xgb_ispu_jatim_multi_otak.pkl'
joblib.dump(paket_model, nama_file)

print(f"Selesai! 6 Model pintar telah dibungkus jadi satu dan diekspor sebagai '{nama_file}'")

# [MLFLOW] Mencatat file bundle pkl ke dalam run terakhir atau run khusus
with mlflow.start_run(run_name="Final_Bundle_Export"):
    mlflow.log_artifact(nama_file)
    print(f"📦 File {nama_file} juga telah diunggah ke MLflow Artifacts.")

Selesai! 6 Model pintar telah dibungkus jadi satu dan diekspor sebagai 'xgb_ispu_jatim_multi_otak.pkl'
📦 File xgb_ispu_jatim_multi_otak.pkl juga telah diunggah ke MLflow Artifacts.
